# Chapter 1 Exercises 4-7

These exercises are checks and analytical comparisons. They do not require an optimizer. We use ordinary matrix calculations and direct SciBmad lattice inspection.

In [1]:
using SciBmad
using GTPSA
using LinearAlgebra
using Printf

function find_tutorial_root()
    candidates = [
        pwd(),
        normpath(joinpath(pwd(), "..")),
        normpath(joinpath(pwd(), "..", "..")),
        normpath(joinpath(pwd(), "..", "..", "..")),
    ]
    for root in candidates
        if isfile(joinpath(root, "chapter01_fodo_scibmad.ipynb")) && isdir(joinpath(root, "lattices", "chapter_1"))
            return root
        end
    end
    error("Cannot locate Ring_Design_Tutorial_SciBmad root.")
end

const tutorial_root = find_tutorial_root()
include(joinpath(tutorial_root, "lattices", "chapter_1", "chapter1_fodoF_solution.jl"))

const L_quad = 0.5
const D1_len = 0.609
const D2_len = 1.241
const B_chord = 6.86
const B_angle = pi / 132
const B_arc = B_chord * B_angle / (2sin(B_angle / 2))

const species_ref = Species("electron")
const E_ref = 18e9

D1() = Drift(name="D1", L=D1_len)
D2() = Drift(name="D2", L=D2_len)
B(name="B") = SBend(name=name, L=B_arc, angle=B_angle)

function build_forward_arc_fodo(kqf, kqd)
    elements = [
        Quadrupole(name="QF", L=L_quad, Kn1=kqf), D1(), B("B1"), D2(),
        Quadrupole(name="QD", L=L_quad, Kn1=kqd), D1(), B("B2"), D2(),
    ]
    beamline = Beamline(elements; species_ref=species_ref, E_ref=E_ref)
    return (; elements, beamline)
end

function build_reverse_arc_fodo(kqf, kqd)
    elements = [
        Quadrupole(name="QF", L=L_quad, Kn1=kqf), D2(), B("B1"), D1(),
        Quadrupole(name="QD", L=L_quad, Kn1=kqd), D2(), B("B2"), D1(),
    ]
    beamline = Beamline(elements; species_ref=species_ref, E_ref=E_ref)
    return (; elements, beamline)
end

fodo0 = build_forward_arc_fodo(0.4, -0.4)


(elements = LineElement[LineElement:
  UniversalParams                        BeamlineParams
   kind            = Quadrupole           beamline_index = 1
   name            = QF                   species_ref    = electron
   L               = 0.5                  E_ref          = 1.8e10
   tracking_method = SciBmadStandard()    s              = 0
                                          s_downstream   = 0.5

  BMultipoleParams
   Kn1   = 0.4

, LineElement:
  UniversalParams                        BeamlineParams
   kind            = Drift                beamline_index = 2
   name            = D1                   species_ref    = electron
   L               = 0.609                E_ref          = 1.8e10
   tracking_method = SciBmadStandard()    s              = 0.5
                                          s_downstream   = 1.109

, LineElement:
  UniversalParams                        BeamlineParams
   kind            = SBend                beamline_index = 3
   name            = B1  

## Shared Transfer-Map Helpers

For the comparison exercises we compute the linear transfer matrix directly by tracking GTPSA variables through the SciBmad beamline. This is ordinary map extraction, not optimization.

In [2]:
function track_a_particle(v0, beamline)
    v = similar(v0)
    v .= v0
    bunch = Bunch(v; species=beamline.species_ref, p_over_q_ref=beamline.p_over_q_ref)
    track!(bunch, beamline)
    return copy(bunch.coords.v)
end

function linear_map(beamline; x0=zeros(6))
    d = Descriptor(6, 2)
    xvars = vars(d)
    v0 = [x0[i] + xvars[i] for i in 1:6]
    vout = track_a_particle(v0, beamline)

    M = zeros(6, 6)
    for i in 1:6, j in 1:6
        powers = zeros(Int, 6)
        powers[j] = 1
        M[i, j] = vout[i][powers]
    end
    return M
end

function stable_phase_advance(M2)
    cos_mu = clamp(0.5 * tr(M2), -1.0, 1.0)
    return acos(cos_mu)
end

function periodic_twiss_from_map(M2)
    mu = stable_phase_advance(M2)
    smu = sin(mu)
    beta = M2[1, 2] / smu
    alpha = (M2[1, 1] - M2[2, 2]) / (2smu)
    return (beta=beta, alpha=alpha, mu=mu)
end


periodic_twiss_from_map (generic function with 1 method)

## Exercise 4: Analytical Thin-Lens FODO

Use a thin-lens model with the same total cell length as the arc FODO. The zero-length focusing and defocusing quadrupoles are separated by half of the cell length.

In this analytical model there is no bending. Therefore there is no extra horizontal focusing from the bends, and we take the two quadrupole integrated strengths to be opposite:

$$
q_D=-q_F.
$$

Let the half-cell drift length be $l$, and define the drift and thin-lens quadrupole matrices by

$$
D(l)=
\begin{pmatrix}
1 & l \\
0 & 1
\end{pmatrix},
\qquad
Q(q)=
\begin{pmatrix}
1 & 0 \\
-q & 1
\end{pmatrix}.
$$

In the horizontal plane, the focusing quadrupole has strength $+q$ and the defocusing quadrupole has strength $-q$. Starting the cell just after the focusing quadrupole, the one-cell transfer matrix is

$$
M_x = Q(q)D(l)Q(-q)D(l).
$$

Multiplying the matrices gives

$$
M_x =
\begin{pmatrix}
1+lq & 2l+l^2q \\
-lq^2 & 1-lq-l^2q^2
\end{pmatrix}.
$$

Therefore the trace of the one-cell transfer matrix is

$$
\mathrm{Tr}(M_x)
=
(1+lq)+(1-lq-l^2q^2)
=2-l^2q^2.
$$

For stable periodic motion,

$$
\cos\mu=\frac{1}{2}\mathrm{Tr}(M_x).
$$

A 90-degree phase advance has $\mu=\pi/2$ and $\cos\mu=0$, so $\mathrm{Tr}(M_x)=0$. Thus

$$
2-l^2q^2=0
\quad\Longrightarrow\quad
q=\frac{\sqrt{2}}{l}.
$$

The vertical plane has the same trace because the signs of the two thin lenses are swapped. To compare with a thick quadrupole strength, divide the integrated strength $q$ by the physical quadrupole length used in the numerical lattice.

In [3]:
drift(L) = [1.0 L; 0.0 1.0]
thin_quad(q) = [1.0 0.0; -q 1.0]

cell_length = 2L_quad + 2(D1_len + B_arc + D2_len)
half_cell_length = cell_length / 2
q_thin = sqrt(2) / half_cell_length
K_equiv = q_thin / L_quad

M_thin_x_at_QF = thin_quad(q_thin) * drift(half_cell_length) * thin_quad(-q_thin) * drift(half_cell_length)
M_thin_y_at_QF = thin_quad(-q_thin) * drift(half_cell_length) * thin_quad(q_thin) * drift(half_cell_length)
M_thin_x_at_QD = thin_quad(-q_thin) * drift(half_cell_length) * thin_quad(q_thin) * drift(half_cell_length)
M_thin_y_at_QD = thin_quad(q_thin) * drift(half_cell_length) * thin_quad(-q_thin) * drift(half_cell_length)

twiss_thin_x_QF = periodic_twiss_from_map(M_thin_x_at_QF)
twiss_thin_y_QF = periodic_twiss_from_map(M_thin_y_at_QF)
twiss_thin_x_QD = periodic_twiss_from_map(M_thin_x_at_QD)
twiss_thin_y_QD = periodic_twiss_from_map(M_thin_y_at_QD)

@printf("Total thick-cell length used for thin model = %.12f m\n", cell_length)
@printf("Half-cell length l = %.12f m\n", half_cell_length)
@printf("Thin integrated strength q = %.12f 1/m\n", q_thin)
@printf("Equivalent thick K1 = q/L_quad = %.12f 1/m^2\n", K_equiv)
@printf("Chapter 1 numerical KQF = %.12f, KQD = %.12f\n", kQF_arc, kQD_arc)
@printf("Thin-lens beta_x at QF = %.12f m, alpha_x = %.12f\n", twiss_thin_x_QF.beta, twiss_thin_x_QF.alpha)
@printf("Thin-lens beta_y at QF = %.12f m, alpha_y = %.12f\n", twiss_thin_y_QF.beta, twiss_thin_y_QF.alpha)
@printf("Thin-lens beta_x at QD = %.12f m, alpha_x = %.12f\n", twiss_thin_x_QD.beta, twiss_thin_x_QD.alpha)
@printf("Thin-lens beta_y at QD = %.12f m, alpha_y = %.12f\n", twiss_thin_y_QD.beta, twiss_thin_y_QD.alpha)


Total thick-cell length used for thin model = 18.420323818702 m
Half-cell length l = 9.210161909351 m
Thin integrated strength q = 0.153549261815 1/m
Equivalent thick K1 = q/L_quad = 0.307098523629 1/m^2
Chapter 1 numerical KQF = 0.312659097023, KQD = -0.312792879287
Thin-lens beta_x at QF = 31.445459702558 m, alpha_x = 2.414213562373
Thin-lens beta_y at QF = 5.395187934846 m, alpha_y = -0.414213562373
Thin-lens beta_x at QD = 5.395187934846 m, alpha_x = -0.414213562373
Thin-lens beta_y at QD = 31.445459702558 m, alpha_y = 2.414213562373


## Exercise 5: Forward and Reverse Arc Cells

The forward and reverse cells use the same quadrupole strengths and total element lengths, but the drift order around each bend is reversed. This changes the periodic beta and alpha functions at the chosen starting point, even though the one-cell phase advance remains the same.

In [4]:
forward_arc = build_forward_arc_fodo(kQF_arc, kQD_arc)
reverse_arc = build_reverse_arc_fodo(kQF_arc, kQD_arc)

M_forward = linear_map(forward_arc.beamline)
M_reverse = linear_map(reverse_arc.beamline)

twx_forward = periodic_twiss_from_map(M_forward[1:2, 1:2])
twy_forward = periodic_twiss_from_map(M_forward[3:4, 3:4])
twx_reverse = periodic_twiss_from_map(M_reverse[1:2, 1:2])
twy_reverse = periodic_twiss_from_map(M_reverse[3:4, 3:4])

println("Using the same strengths:")
@printf("  KQF = %.15f\n", kQF_arc)
@printf("  KQD = %.15f\n", kQD_arc)

println("\nForward arc periodic Twiss:")
@printf("  beta_x = %.10f, alpha_x = %.10f, mu_x = %.12f deg\n", twx_forward.beta, twx_forward.alpha, rad2deg(twx_forward.mu))
@printf("  beta_y = %.10f, alpha_y = %.10f, mu_y = %.12f deg\n", twy_forward.beta, twy_forward.alpha, rad2deg(twy_forward.mu))

println("\nReverse arc periodic Twiss:")
@printf("  beta_x = %.10f, alpha_x = %.10f, mu_x = %.12f deg\n", twx_reverse.beta, twx_reverse.alpha, rad2deg(twx_reverse.mu))
@printf("  beta_y = %.10f, alpha_y = %.10f, mu_y = %.12f deg\n", twy_reverse.beta, twy_reverse.alpha, rad2deg(twy_reverse.mu))


Using the same strengths:
  KQF = 0.312659097022843
  KQD = -0.312792879287368

Forward arc periodic Twiss:
  beta_x = 30.6240861316, alpha_x = -2.4014718318, mu_x = 90.000000018742 deg
  beta_y = 5.5490297918, alpha_y = 0.4766729386, mu_y = 90.000000001744 deg

Reverse arc periodic Twiss:
  beta_x = 30.6241751724, alpha_x = -2.4012984144, mu_x = 90.000000018742 deg
  beta_y = 5.5490297918, alpha_y = 0.4766729386, mu_y = 90.000000001744 deg


## Exercise 6: Arc Length and Chord Length

The Chapter 1 input bend length is a chord length. The `SBend` model uses the true arc length:

$$
L_{\mathrm{arc}} = L_{\mathrm{chord}}\frac{\theta}{2\sin(\theta/2)}.
$$

For small bend angle $\theta$, $2\sin(\theta/2)\approx\theta$, so the two lengths become nearly identical.

In [6]:
const B_chord = 6.86
const B_angle = pi / 132
const B_arc = B_chord * B_angle / (2sin(B_angle / 2))
arc_minus_chord = B_arc - B_chord
relative_difference = arc_minus_chord / B_chord

@printf("Bend angle theta = %.15e rad\n", B_angle)
@printf("Chord length     = %.15f m\n", B_chord)
@printf("Arc length       = %.15f m\n", B_arc)
@printf("Difference       = %.15e m\n", arc_minus_chord)
@printf("Relative diff    = %.15e\n", relative_difference)


Bend angle theta = 2.379994434537722e-02 rad
Chord length     = 6.860000000000000 m
Arc length       = 6.860161909351031 m
Difference       = 1.619093510303315e-04 m
Relative diff    = 2.360194621433404e-05


## Exercise 7: Inspect the SciBmad Lattice

We inspect the initial forward FODO cell `fodo0`, list its elements, and confirm the total cell length. The optics table is evaluated at the entrance and at the downstream end of each element, so its longitudinal positions should match `0` followed by the cumulative element lengths.

In [7]:
function safe_property(ele, name, default=missing)
    try
        return getproperty(ele, name)
    catch
        return default
    end
end

function element_kind(ele)
    return string(safe_property(ele, :kind, nameof(typeof(ele))))
end

function bend_angle(ele)
    kind = element_kind(ele)
    if occursin("Bend", kind)
        return safe_property(ele, :g_ref, 0.0) * safe_property(ele, :L, 0.0)
    end
    return 0.0
end

println("index  name  kind          L [m]          Kn1             angle [rad]")
println("-----  ----  ------------  -------------  --------------  -------------")

for (i, ele) in enumerate(fodo0.elements)
    @printf(
        "%5d  %-4s  %-12s  %13.9f  %14.9f  %13.9f\n",
        i,
        string(safe_property(ele, :name, "")),
        element_kind(ele),
        safe_property(ele, :L, 0.0),
        safe_property(ele, :Kn1, 0.0),
        bend_angle(ele),
    )
end

lengths = [safe_property(ele, :L, 0.0) for ele in fodo0.elements]
s_positions = vcat(0.0, cumsum(lengths))

println("\nTotal cell length = ", sum(lengths), " m")
println("Optics table evaluation positions, if evaluated at element ends:")
for (i, s) in enumerate(s_positions)
    @printf("  row %2d: s = %.9f m\n", i, s)
end


index  name  kind          L [m]          Kn1             angle [rad]
-----  ----  ------------  -------------  --------------  -------------
    1  QF    Quadrupole      0.500000000     0.400000000    0.000000000
    2  D1    Drift           0.609000000     0.000000000    0.000000000
    3  B1    SBend           6.860161909     0.000000000    0.023799944
    4  D2    Drift           1.241000000     0.000000000    0.000000000
    5  QD    Quadrupole      0.500000000    -0.400000000    0.000000000
    6  D1    Drift           0.609000000     0.000000000    0.000000000
    7  B2    SBend           6.860161909     0.000000000    0.023799944
    8  D2    Drift           1.241000000     0.000000000    0.000000000

Total cell length = 18.420323818702062 m
Optics table evaluation positions, if evaluated at element ends:
  row  1: s = 0.000000000 m
  row  2: s = 0.500000000 m
  row  3: s = 1.109000000 m
  row  4: s = 7.969161909 m
  row  5: s = 9.210161909 m
  row  6: s = 9.710161909 m
  row  